In [25]:
import math
import numpy as np
import matplotlib as plt

In [ ]:
class Value:
    
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backword = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label 

    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backword():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backword = _backword
        return out
    
    def __neg__(self):
        return self.data * -1
    
    def __sub__(self, other):
        return self.data + (-other)
    
    def __rsub__(self, other):
        return self - other

    def __radd__(self, other):
        return self + other
    
    def __mul__(self, other):
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "-")
        def _backword():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        self._backword = _backword
        return out
    
    def __rmul__(self, other):
        return self * Value(other)
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self, other), f"^{other}")
        def _backword():
            self.grad += other * (self.data ** (other -1)) * out.grad
        self._backword = _backword
        return out
    
    def __truediv__(self, other):
        return self * other ** -1
    
    def __rtruediv__(self, other):
        return self / other
    
    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,), "e")
        def _backword():
            self.grad += out.data * out.grad
        self._backword = _backword
        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(x*2)-1)/(math.exp(x*2)+1)
        out = Value(t, (self,), "tanh")
        def _backword():
            self.grad += (1-t**2) * out.grad
        out._backword = _backword
        return out
    
    def backword(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backword()

In [31]:
a = Value(2.0)
b = Value(-3.0)
c = Value(4.0)

d = a*b + c
d

Value(data=-2.0)